# Solution A — Demo / Inference

Loads the trained Solution A model bundle and generates predictions for any CSV file containing `premise` and `hypothesis` columns. Set `DEMO_INPUT_PATH` to your input file and run all cells.

In [10]:
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'scipy': 'scipy',
    'sklearn': 'scikit-learn',
    'joblib': 'joblib',
}

missing_packages = [
    pip_name
    for module_name, pip_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    print('Installing missing packages:', missing_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing_packages])
else:
    print('All required packages are already available.')


All required packages are already available.


## 1. Imports and path resolution

The notebook should work whether it is run from the repository root or from inside `solution-a/`.

The path helper below searches upward until it finds the coursework root that contains both:

- `training_data/NLI/train.csv`
- `nlu_bundle-feature-unified-local-scorer/`

This avoids the broken relative-path problem that the earlier notebook had.


In [11]:
from __future__ import annotations

import re
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import MaxAbsScaler, StandardScaler
from sklearn.svm import LinearSVC

SEED = 42


def find_project_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (
            (candidate / 'training_data' / 'NLI' / 'train.csv').exists()
            and (candidate / 'nlu_bundle-feature-unified-local-scorer').exists()
        ):
            return candidate
    raise FileNotFoundError(
        'Could not find the coursework root. Expected to find training_data/NLI/train.csv '
        'and nlu_bundle-feature-unified-local-scorer/ in the same project tree.'
    )


PROJECT_ROOT = find_project_root()
NOTEBOOK_DIR = PROJECT_ROOT / 'solution-a'
TRAIN_PATH = PROJECT_ROOT / 'training_data' / 'NLI' / 'train.csv'
DEV_PATH = PROJECT_ROOT / 'training_data' / 'NLI' / 'dev.csv'
TRIAL_PATH = PROJECT_ROOT / 'trial_data' / 'NLI_trial.csv'
LOCAL_SCORER_ROOT = PROJECT_ROOT / 'nlu_bundle-feature-unified-local-scorer'
OFFICIAL_BASELINE_PATH = LOCAL_SCORER_ROOT / 'baseline' / '25_DEV_NLI.csv'
ARTEFACT_DIR = NOTEBOOK_DIR / 'artifacts_solution_a'
ARTEFACT_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('PROJECT_ROOT =', PROJECT_ROOT)
print('TRAIN_PATH   =', TRAIN_PATH)
print('DEV_PATH     =', DEV_PATH)
print('TRIAL_PATH   =', TRIAL_PATH)
print('ARTEFACT_DIR =', ARTEFACT_DIR)


PROJECT_ROOT = /Users/jiho/Documents/YR3/34812NLU/NLU_CW
TRAIN_PATH   = /Users/jiho/Documents/YR3/34812NLU/NLU_CW/training_data/NLI/train.csv
DEV_PATH     = /Users/jiho/Documents/YR3/34812NLU/NLU_CW/training_data/NLI/dev.csv
TRIAL_PATH   = /Users/jiho/Documents/YR3/34812NLU/NLU_CW/trial_data/NLI_trial.csv
ARTEFACT_DIR = /Users/jiho/Documents/YR3/34812NLU/NLU_CW/solution-a/artifacts_solution_a


## Path setup

Paths are resolved automatically from the project root. The bundle must exist at `artifacts_solution_a/nli_solution_a_bundle.joblib`. Run `solution_A_train.ipynb` first if it does not.

In [12]:
# Bundle is stored alongside this notebook in artifacts_solution_a/
# This works whether run locally or by the marking team.
NOTEBOOK_DIR = Path(__file__).parent if '__file__' in dir() else Path.cwd()
ARTEFACT_DIR = NOTEBOOK_DIR / 'artifacts_solution_a'

# Fallback: if bundle not found locally, search the project tree
SOLUTION_A_BUNDLE_PATH = ARTEFACT_DIR / 'nli_solution_a_bundle.joblib'
if not SOLUTION_A_BUNDLE_PATH.exists():
    try:
        PROJECT_ROOT = find_project_root()
        SOLUTION_A_BUNDLE_PATH = PROJECT_ROOT / 'solution-a' / 'artifacts_solution_a' / 'nli_solution_a_bundle.joblib'
        ARTEFACT_DIR = SOLUTION_A_BUNDLE_PATH.parent
    except FileNotFoundError:
        pass

print('Bundle path:', SOLUTION_A_BUNDLE_PATH)
print('Bundle exists:', SOLUTION_A_BUNDLE_PATH.exists())

Bundle path: /Users/jiho/Documents/YR3/34812NLU/NLU_CW/code/artifacts_solution_a/nli_solution_a_bundle.joblib
Bundle exists: True


## 5. Final Solution A representation

The main Solution A representation stays inside traditional machine learning, but is richer than the internal baseline because it adds pairwise structure.

Feature blocks used by the rich representation:

1. **Premise word TF-IDF**: word 1-2 grams from the premise only
2. **Hypothesis word TF-IDF**: word 1-2 grams from the hypothesis only
3. **Shared-space interactions**: absolute difference and element-wise product after projecting both texts into the same word-TF-IDF space
4. **Pair-level character TF-IDF**: character n-grams from `premise [SEP] hypothesis`
5. **Hand-crafted pair features** computed directly from the provided text:
   - lexical overlap
   - new-token ratio
   - length features
   - negation mismatch
   - number mismatch
   - simple punctuation cues

The richer representation supports several Category A candidates: Logistic Regression, Linear SVM, ablations, bootstrap LR ensembles, and vocabulary-size sensitivity checks.


In [13]:
TOKEN_PATTERN = re.compile(r"[a-z0-9]+(?:'[a-z0-9]+)?")
NUMBER_PATTERN = re.compile(r'\d+(?:\.\d+)?')
NEGATION_TOKENS = {'no', 'not', 'never', 'none', 'nobody', 'nothing', 'neither', 'nor', 'without'}
FULL_FEATURE_BLOCK_ORDER = [
    'premise_word_tfidf',
    'hypothesis_word_tfidf',
    'shared_abs_difference',
    'shared_product',
    'pair_char_tfidf',
    'handcrafted_dense',
]


def normalize_text(text: str) -> str:
    text = str(text)
    text = text.replace('’', "'").replace('‘', "'")
    text = text.replace('“', '"').replace('”', '"')
    return re.sub(r'\s+', ' ', text).strip()


def tokenize(text: str) -> list[str]:
    return TOKEN_PATTERN.findall(normalize_text(text).lower())


HANDCRAFTED_FEATURE_NAMES = [
    'hypothesis_token_recall',
    'premise_token_precision',
    'jaccard',
    'new_token_ratio',
    'premise_length',
    'hypothesis_length',
    'length_ratio',
    'length_difference',
    'premise_has_negation',
    'hypothesis_has_negation',
    'negation_mismatch',
    'shared_number_count',
    'number_mismatch',
    'exact_string_match',
    'hypothesis_token_subset',
    'premise_has_question_mark',
    'hypothesis_has_question_mark',
    'question_mark_delta',
    'premise_has_exclamation_mark',
    'hypothesis_has_exclamation_mark',
    'exclamation_mark_delta',
]


def build_handcrafted_pair_features(df: pd.DataFrame) -> np.ndarray:
    rows = []
    for premise, hypothesis in zip(df['premise'], df['hypothesis']):
        premise_text = normalize_text(premise)
        hypothesis_text = normalize_text(hypothesis)
        premise_tokens = tokenize(premise_text)
        hypothesis_tokens = tokenize(hypothesis_text)
        premise_set = set(premise_tokens)
        hypothesis_set = set(hypothesis_tokens)

        overlap = len(premise_set & hypothesis_set)
        union = len(premise_set | hypothesis_set)
        hypothesis_token_recall = overlap / len(hypothesis_set) if hypothesis_set else 0.0
        premise_token_precision = overlap / len(premise_set) if premise_set else 0.0
        jaccard = overlap / union if union else 0.0
        new_token_ratio = len(hypothesis_set - premise_set) / len(hypothesis_set) if hypothesis_set else 0.0

        premise_length = len(premise_tokens)
        hypothesis_length = len(hypothesis_tokens)
        length_ratio = hypothesis_length / premise_length if premise_length else 0.0
        length_difference = premise_length - hypothesis_length

        premise_has_negation = int(any(tok in NEGATION_TOKENS or tok.endswith("n't") for tok in premise_tokens))
        hypothesis_has_negation = int(any(tok in NEGATION_TOKENS or tok.endswith("n't") for tok in hypothesis_tokens))
        negation_mismatch = int(premise_has_negation != hypothesis_has_negation)

        premise_numbers = NUMBER_PATTERN.findall(premise_text)
        hypothesis_numbers = NUMBER_PATTERN.findall(hypothesis_text)
        shared_number_count = len(set(premise_numbers) & set(hypothesis_numbers))
        number_mismatch = int(bool(premise_numbers or hypothesis_numbers) and set(premise_numbers) != set(hypothesis_numbers))

        exact_string_match = int(premise_text.lower() == hypothesis_text.lower())
        hypothesis_token_subset = int(hypothesis_set.issubset(premise_set)) if hypothesis_set else 0

        premise_has_question_mark = int('?' in premise_text)
        hypothesis_has_question_mark = int('?' in hypothesis_text)
        question_mark_delta = hypothesis_has_question_mark - premise_has_question_mark
        premise_has_exclamation_mark = int('!' in premise_text)
        hypothesis_has_exclamation_mark = int('!' in hypothesis_text)
        exclamation_mark_delta = hypothesis_has_exclamation_mark - premise_has_exclamation_mark

        rows.append([
            hypothesis_token_recall,
            premise_token_precision,
            jaccard,
            new_token_ratio,
            premise_length,
            hypothesis_length,
            length_ratio,
            length_difference,
            premise_has_negation,
            hypothesis_has_negation,
            negation_mismatch,
            shared_number_count,
            number_mismatch,
            exact_string_match,
            hypothesis_token_subset,
            premise_has_question_mark,
            hypothesis_has_question_mark,
            question_mark_delta,
            premise_has_exclamation_mark,
            hypothesis_has_exclamation_mark,
            exclamation_mark_delta,
        ])

    return np.asarray(rows, dtype=np.float32)


def sparse_absolute_difference(left: sp.csr_matrix, right: sp.csr_matrix) -> sp.csr_matrix:
    diff = (left - right).tocsr(copy=True)
    diff.data = np.abs(diff.data)
    return diff


In [14]:
class SolutionAFeatureBuilder:
    def __init__(
        self,
        premise_word_max_features: int = 12000,
        hypothesis_word_max_features: int = 12000,
        shared_word_max_features: int = 8000,
        pair_char_max_features: int = 8000,
    ) -> None:
        self.premise_word_vectorizer = TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_features=premise_word_max_features,
            sublinear_tf=True,
        )
        self.hypothesis_word_vectorizer = TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_features=hypothesis_word_max_features,
            sublinear_tf=True,
        )
        self.shared_word_vectorizer = TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_features=shared_word_max_features,
            sublinear_tf=True,
        )
        self.pair_char_vectorizer = TfidfVectorizer(
            analyzer='char_wb',
            ngram_range=(3, 5),
            min_df=2,
            max_features=pair_char_max_features,
            sublinear_tf=True,
        )
        self.handcrafted_scaler = StandardScaler()
        self.tfidf_scaler = MaxAbsScaler()
        self.handcrafted_feature_names_ = HANDCRAFTED_FEATURE_NAMES.copy()
        self.feature_block_dimensions_ = {}
        self.config_ = {
            'premise_word_max_features': premise_word_max_features,
            'hypothesis_word_max_features': hypothesis_word_max_features,
            'shared_word_max_features': shared_word_max_features,
            'pair_char_max_features': pair_char_max_features,
        }

    def _normalised_premise_series(self, df: pd.DataFrame) -> pd.Series:
        return df['premise'].fillna('').map(normalize_text)

    def _normalised_hypothesis_series(self, df: pd.DataFrame) -> pd.Series:
        return df['hypothesis'].fillna('').map(normalize_text)

    def fit(self, df: pd.DataFrame) -> 'SolutionAFeatureBuilder':
        premise_text = self._normalised_premise_series(df)
        hypothesis_text = self._normalised_hypothesis_series(df)
        pair_text = premise_text + ' [SEP] ' + hypothesis_text
        handcrafted = build_handcrafted_pair_features(df)

        self.premise_word_vectorizer.fit(premise_text)
        self.hypothesis_word_vectorizer.fit(hypothesis_text)
        self.shared_word_vectorizer.fit(pd.concat([premise_text, hypothesis_text], ignore_index=True))
        self.pair_char_vectorizer.fit(pair_text)
        self.handcrafted_scaler.fit(handcrafted)

        # Fit MaxAbsScaler on concatenated TF-IDF blocks so all sparse features
        # are on the same scale as the StandardScaler-normalised handcrafted features.
        _shared_prem = self.shared_word_vectorizer.transform(premise_text)
        _shared_hyp = self.shared_word_vectorizer.transform(hypothesis_text)
        _tfidf_combined = sp.hstack([
            self.premise_word_vectorizer.transform(premise_text),
            self.hypothesis_word_vectorizer.transform(hypothesis_text),
            sparse_absolute_difference(_shared_prem, _shared_hyp),
            _shared_prem.multiply(_shared_hyp),
            self.pair_char_vectorizer.transform(pair_text),
        ], format='csr')
        self.tfidf_scaler.fit(_tfidf_combined)

        self.feature_block_dimensions_ = {
            'premise_word_tfidf': len(self.premise_word_vectorizer.get_feature_names_out()),
            'hypothesis_word_tfidf': len(self.hypothesis_word_vectorizer.get_feature_names_out()),
            'shared_abs_difference': len(self.shared_word_vectorizer.get_feature_names_out()),
            'shared_product': len(self.shared_word_vectorizer.get_feature_names_out()),
            'pair_char_tfidf': len(self.pair_char_vectorizer.get_feature_names_out()),
            'handcrafted_dense': len(self.handcrafted_feature_names_),
        }
        return self

    def transform(self, df: pd.DataFrame) -> sp.csr_matrix:
        blocks = build_feature_block_matrices(feature_components_from_builder(self), df)
        return stack_selected_blocks(blocks, FULL_FEATURE_BLOCK_ORDER)

    def fit_transform(self, df: pd.DataFrame) -> sp.csr_matrix:
        self.fit(df)
        return self.transform(df)


def feature_components_from_builder(feature_builder: SolutionAFeatureBuilder) -> dict:
    return {
        'premise_word_vectorizer': feature_builder.premise_word_vectorizer,
        'hypothesis_word_vectorizer': feature_builder.hypothesis_word_vectorizer,
        'shared_word_vectorizer': feature_builder.shared_word_vectorizer,
        'pair_char_vectorizer': feature_builder.pair_char_vectorizer,
        'handcrafted_scaler': feature_builder.handcrafted_scaler,
        'tfidf_scaler': feature_builder.tfidf_scaler,
        'handcrafted_feature_names': feature_builder.handcrafted_feature_names_,
        'feature_block_dimensions': feature_builder.feature_block_dimensions_,
        'feature_config': feature_builder.config_,
    }


def build_feature_block_matrices(feature_components: dict, df: pd.DataFrame) -> dict[str, sp.csr_matrix]:
    premise_text = df['premise'].fillna('').map(normalize_text)
    hypothesis_text = df['hypothesis'].fillna('').map(normalize_text)
    pair_text = premise_text + ' [SEP] ' + hypothesis_text

    premise_word = feature_components['premise_word_vectorizer'].transform(premise_text)
    hypothesis_word = feature_components['hypothesis_word_vectorizer'].transform(hypothesis_text)

    shared_premise = feature_components['shared_word_vectorizer'].transform(premise_text)
    shared_hypothesis = feature_components['shared_word_vectorizer'].transform(hypothesis_text)
    shared_abs_difference = sparse_absolute_difference(shared_premise, shared_hypothesis)
    shared_product = shared_premise.multiply(shared_hypothesis)

    pair_char = feature_components['pair_char_vectorizer'].transform(pair_text)
    handcrafted = build_handcrafted_pair_features(df)
    handcrafted_scaled = feature_components['handcrafted_scaler'].transform(handcrafted)

    if 'tfidf_scaler' in feature_components:
        _tfidf_combined = sp.hstack(
            [premise_word, hypothesis_word, shared_abs_difference, shared_product, pair_char],
            format='csr',
        )
        _tfidf_scaled = feature_components['tfidf_scaler'].transform(_tfidf_combined)
        _splits = [
            premise_word.shape[1],
            premise_word.shape[1] + hypothesis_word.shape[1],
            premise_word.shape[1] + hypothesis_word.shape[1] + shared_abs_difference.shape[1],
            premise_word.shape[1] + hypothesis_word.shape[1] + 2 * shared_abs_difference.shape[1],
        ]
        premise_word = _tfidf_scaled[:, :_splits[0]]
        hypothesis_word = _tfidf_scaled[:, _splits[0]:_splits[1]]
        shared_abs_difference = _tfidf_scaled[:, _splits[1]:_splits[2]]
        shared_product = _tfidf_scaled[:, _splits[2]:_splits[3]]
        pair_char = _tfidf_scaled[:, _splits[3]:]

    return {
        'premise_word_tfidf': premise_word,
        'hypothesis_word_tfidf': hypothesis_word,
        'shared_abs_difference': shared_abs_difference,
        'shared_product': shared_product,
        'pair_char_tfidf': pair_char,
        'handcrafted_dense': sp.csr_matrix(handcrafted_scaled),
    }


def stack_selected_blocks(blocks: dict[str, sp.csr_matrix], selected_blocks: list[str]) -> sp.csr_matrix:
    return sp.hstack([blocks[block_name] for block_name in selected_blocks], format='csr')


def transform_with_feature_components(
    feature_components: dict,
    df: pd.DataFrame,
    selected_blocks: list[str] | None = None,
) -> sp.csr_matrix:
    blocks = build_feature_block_matrices(feature_components, df)
    if selected_blocks is None:
        selected_blocks = FULL_FEATURE_BLOCK_ORDER
    return stack_selected_blocks(blocks, selected_blocks)


## 11. Demo / inference mode

This section is intentionally separated from training.

Use it after the artefact bundle exists. It accepts a CSV that contains at least:

- `premise`
- `hypothesis`

If the input also contains `label` (for example the trial file), the helper prints a quick sanity-check metric summary.

For final coursework submission, rename the final test predictions to the required format:

- `Group_n_A.csv`


In [15]:
def read_pair_dataframe(csv_path: Path, require_label: bool) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df.columns = [str(col).lstrip('\ufeff').strip() for col in df.columns]

    required_columns = {'premise', 'hypothesis'}
    missing_required = required_columns - set(df.columns)
    if missing_required:
        raise ValueError(f'{csv_path} is missing required columns: {sorted(missing_required)}')

    if require_label and 'label' not in df.columns:
        raise ValueError(f'{csv_path} must contain a label column for this section of the notebook.')

    if 'label' in df.columns:
        df['label'] = df['label'].astype(int)

    return df


In [16]:
def load_solution_a_bundle(bundle_path: Path = SOLUTION_A_BUNDLE_PATH) -> dict:
    return joblib.load(bundle_path)


def predict_from_solution_a_bundle(bundle: dict, input_df: pd.DataFrame) -> np.ndarray:
    X_input = transform_with_feature_components(
        bundle['feature_components'],
        input_df,
        selected_blocks=bundle['selected_blocks'],
    )

    if bundle['predictor_type'] == 'single_estimator':
        return bundle['estimator'].predict(X_input).astype(int)

    if bundle['predictor_type'] == 'bootstrap_lr_ensemble':
        probability_sum = None
        for estimator in bundle['estimators']:
            model_proba = estimator.predict_proba(X_input)
            probability_sum = model_proba if probability_sum is None else probability_sum + model_proba
        average_proba = probability_sum / len(bundle['estimators'])
        return bundle['classes'][np.argmax(average_proba, axis=1)].astype(int)

    raise ValueError(f"Unsupported predictor type: {bundle['predictor_type']}")


def predict_with_solution_a(
    input_csv: Path,
    output_csv: Path,
    bundle_path: Path = SOLUTION_A_BUNDLE_PATH,
) -> tuple[pd.DataFrame, np.ndarray, Path]:
    bundle = load_solution_a_bundle(bundle_path)
    input_df = read_pair_dataframe(Path(input_csv), require_label=False)
    predictions = predict_from_solution_a_bundle(bundle, input_df)

    output_csv = Path(output_csv)
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    pd.Series(predictions, name='label').to_csv(output_csv, index=False, header=False)

    return input_df, predictions, output_csv


## Specify input and output paths, then run inference

In [17]:
# ── USER CONFIGURATION ──────────────────────────────────────────────────────
# Set DEMO_INPUT_PATH to the CSV file containing test data
# (columns: premise, hypothesis — no label column required).
# Set DEMO_OUTPUT_PATH to where predictions should be written.

DEMO_INPUT_PATH = PROJECT_ROOT / "test_data" / "NLI" / "test.csv"
DEMO_OUTPUT_PATH = ARTEFACT_DIR / 'nli_solution_a_test_predictions.csv'
# ────────────────────────────────────────────────────────────────────────────

In [18]:
if not DEMO_INPUT_PATH.exists():
    print(f'Input file not found: {DEMO_INPUT_PATH}')
    print('Set DEMO_INPUT_PATH to a CSV with premise and hypothesis columns.')
else:
    demo_df, demo_predictions, demo_output_path = predict_with_solution_a(
        input_csv=DEMO_INPUT_PATH,
        output_csv=DEMO_OUTPUT_PATH,
    )
    print('Predictions written to:', demo_output_path)
    print('Number of predictions:', len(demo_predictions))
    print('Prediction distribution:', pd.Series(demo_predictions).value_counts().to_dict())

    if 'label' in demo_df.columns:
        sanity_metrics = metric_summary(demo_df['label'].values, demo_predictions)
        print('\nSanity-check metrics (labels available):')
        pd.DataFrame([sanity_metrics]).T

Predictions written to: /Users/jiho/Documents/YR3/34812NLU/NLU_CW/code/artifacts_solution_a/nli_solution_a_test_predictions.csv
Number of predictions: 3302
Prediction distribution: {0: 1735, 1: 1567}
